# Step 2 — Build Claim Feature Matrix
**Roadmap reference:** Data engineering — join claim header + assignment history + cause tags

## Goal
Assemble one flat analytical record per claim.  This is the training and analysis
dataset for every step that follows.

## Features derived
| Feature | Source | Description |
|---------|--------|-------------|
| assignment_count | History | Total transitions |
| pct_manual | History | Share of manual-trigger events |
| pct_system | History | Share of system-trigger events |
| pct_user_auto | History | Share of user-automated events |
| same_adjuster_returned | History | Did a prior adjuster reappear? (User-3 pattern) |
| n_unique_adjusters | History | How many distinct adjusters handled the claim |
| avg_hrs_between_events | History | Velocity of handoffs |
| final_adjuster_appeared_earlier | History | Final owner was already involved earlier |
| geographic_mismatch | Claims | loss_state != insured_state |
| exposure_added_within_48h | Claims | Complexity signal available at FNOL+48h |

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

DATA = Path("data")

claims  = pd.read_csv(DATA / "cp_claims_2024.csv")
history = pd.read_csv(DATA / "step1_tagged_assignments.csv")
history["timestamp_dt"] = pd.to_datetime(history["timestamp"], format="%m/%d/%Y %I:%M %p")

print(f"Claims: {len(claims)} | History: {len(history)}")

Claims: 200 | History: 1116


In [2]:
# ── Derive assignment-level features per claim ────────────────────────────────
def build_features(grp):
    grp = grp.sort_values("assignment_num")
    n = len(grp)
    triggers = grp["trigger"].str.lower()
    n_sys    = (triggers == "system").sum()
    n_man    = (triggers == "manual").sum()
    n_ua     = (grp["cause_tag_refined"] == "User Automated").sum()

    adjs = grp["adjuster_id"].tolist()
    n_unique = grp["adjuster_id"].nunique()
    final_adj = adjs[-1]
    returned = int(final_adj in adjs[:-1]) if n > 1 else 0

    ts = grp["timestamp_dt"]
    diffs = ts.diff().dt.total_seconds() / 3600
    avg_hrs = diffs.mean() if n > 1 else 0.0

    return pd.Series({
        "assignment_count":               n,
        "pct_manual":                     round(n_man / n, 3),
        "pct_system":                     round(n_sys / n, 3),
        "pct_user_auto":                  round(n_ua  / n, 3),
        "n_unique_adjusters":             n_unique,
        "same_adjuster_returned":         returned,
        "final_adjuster_appeared_earlier": returned,
        "avg_hrs_between_events":         round(avg_hrs, 1),
    })

agg = history.groupby("claim_id").apply(build_features).reset_index()
matrix = claims.merge(agg, on="claim_id", how="left")

# Group label encoding for ML (0=0, 1=A, 2=B, 3=C)
matrix["group_label"] = matrix["final_group"].map({"0": 0, "A": 1, "B": 2, "C": 3})

matrix.to_csv(DATA / "step2_feature_matrix.csv", index=False)
print(f"Feature matrix: {len(matrix)} rows × {len(matrix.columns)} columns")
matrix.head()

Feature matrix: 200 rows × 26 columns


C:\Users\SujaySunilNagvekar\AppData\Local\Temp\ipykernel_118056\1879641219.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  agg = history.groupby("claim_id").apply(build_features).reset_index()


,claim_id,lob,loss_cause,policy_type,loss_state,insured_state,geographic_mismatch,stp_flag,reported_loss_amount,n_exposures_at_fnol,...,complexity,assignment_count,pct_manual,pct_system,pct_user_auto,n_unique_adjusters,same_adjuster_returned,final_adjuster_appeared_earlier,avg_hrs_between_events,group_label
0,CP-0001,Commercial Property,Flood - Sewer Backup,BOP,MD,IL,1,Y,120500.0,2,...,High,3.0,0.333,0.667,0.000,3.0,0.0,0.0,88.0,2
1,CP-0002,Commercial Property,Equipment Breakdown,Standalone Property,WV,IN,1,N,42600.0,1,...,Medium,1.0,0.000,1.000,0.000,1.0,0.0,0.0,0.0,0
2,CP-0003,Commercial Property,Vandalism,Standalone Property,PA,VA,1,N,22300.0,1,...,Medium,4.0,0.750,0.250,0.000,4.0,0.0,0.0,102.3,2
3,CP-0004,Commercial Property,Business Interruption,Commercial Package,NY,VA,1,N,277400.0,1,...,High,4.0,0.500,0.500,0.000,3.0,1.0,1.0,104.3,2
4,CP-0005,Commercial Property,Flood - Sewer Backup,BOP,IL,PA,1,N,76100.0,3,...,High,11.0,0.455,0.545,0.182,6.0,1.0,1.0,83.6,3


## Distribution of assignment counts by group

In [3]:
fig = px.box(matrix, x="final_group", y="assignment_count",
             color="final_group",
             title="Assignment Count Distribution by Group",
             labels={"final_group": "Group", "assignment_count": "# Assignments"},
             category_orders={"final_group": ["0", "A", "B", "C"]})
fig.show()

print("\nFTR rate:", round(len(matrix[matrix["final_group"]=="0"]) / len(matrix) * 100, 1), "%")
print("Group C rate:", round(len(matrix[matrix["final_group"]=="C"]) / len(matrix) * 100, 1), "%")


FTR rate: 23.0 %
Group C rate: 56.0 %


## Correlation — feature vs group label

In [4]:
num_cols = ["assignment_count", "pct_manual", "pct_user_auto",
            "n_unique_adjusters", "same_adjuster_returned",
            "avg_hrs_between_events", "geographic_mismatch",
            "exposure_added_within_48h", "reported_loss_amount", "group_label"]
corr = matrix[num_cols].corr()[["group_label"]].drop("group_label").round(3)
corr.columns = ["Correlation with group_label"]
print(corr.sort_values("Correlation with group_label", ascending=False).to_string())

                           Correlation with group_label
n_unique_adjusters                                0.873
assignment_count                                  0.847
avg_hrs_between_events                            0.743
same_adjuster_returned                            0.557
pct_manual                                        0.430
pct_user_auto                                     0.353
exposure_added_within_48h                         0.130
reported_loss_amount                              0.061
geographic_mismatch                               0.008
